# pandaSim: tour of the project

Box manipulation with environmental contact on a Franka Panda: a start and a goal pose of the box on the table give a
plan of primitive motions (slides, pivots, gait steps), which the executor runs with the contact QP of the paper
(Sec. 6). This notebook goes through every part implemented so far, in the order of the pipeline, and runs the
experiments of the paper (Sec. 7) **in simulation**. Swift is shown inside the output cells (`notebook=True`).

**No robot here.** Nothing in this notebook connects to the robot. The same planner and executor run on the robot only
through `experiments/run_box.py --execute`, which you start yourself (`docs/HARDWARE_DAY.md`).

| Sec. | Part | Code |
|---|---|---|
| 0 | Setup and helpers | |
| 1 | The box, its faces and the marked corner | `run_box.parse_pose`, `run_plan.box_pose` |
| 2 | Primitive motions: slide and pivot (paper Sec. 7.2) | `planning/box_primitives.py`, `control/primitive_executor.py` |
| 3 | Inside a pivot: contact QP and the bearing angle $\phi$ | `control/contact_qp.py`, `planning/clearance.py` |
| 4 | Sequencing with a marked box (Sec. 7.4) | `planning/sequencer.py` |
| 5 | Object gaiting with one arm (Sec. 7.3) | `planning/gait.py` |
| 6 | Free rotation against the rigid grasp (Sec. 7.5) | `run_box.predicted_stop` |
| 6b | Two pivots in one grasp: free rotation completes, the rigid grasp stops | `planning/chain.py`, `run_box.py --chain` |
| 7 | Pre-flight and the lag of the joint controller | `run_box.preflight`, `run_box.simulate_tracking` |
| 8 | The robot runs of the paper (protocol) | `configs/protocol.yaml`, `protocol.py` |
| 9 | After a robot run | `calibrate_box.py`, `hardware_tables.py`, `analyze_run.py`, `paper_figure.py` |
| 10 | **Playground**: your own task and parameters | |
| 11 | What the code review of 2026-11-24/25 changed, and how to check it | `pandaSim_handoff/review_2026-11-24/` |

Kernel: the repository's `.venv` (`uv sync --extra sim --extra robot --extra dev`). Running the whole notebook takes
about 15 min; each section can also be run on its own after Sec. 0.

## 0. Setup and helpers

Every task below goes through `run_box.plan_and_simulate`, the function that `run_box.py` calls before it moves the
robot: plan with the feasibility model of the config, then run the plan with the executor on the simulated arm at
1 kHz. The config is `configs/box_real.yaml` (the robot config: marked box, `hand_model: box`, safety thresholds).

`NOTEBOOK = False` opens Swift in a browser tab instead of the cell (use it if the frame stays blank: the frame points
at `localhost`, so the notebook must be opened in a browser on the machine that runs the kernel). In the Swift view:
drag to orbit, right-drag to pan, scroll to zoom. Blue: the box, green ghost: its goal, red sphere: the marked corner.

In [ ]:
import math
import sys
from pathlib import Path

sys.path = [p for p in sys.path if "/opt/ros" not in p]     # ROS Humble's numpy 1.x would shadow the env
import matplotlib.pyplot as plt
import numpy as np
import yaml

HERE = Path.cwd() if (Path.cwd() / "run_box.py").exists() else Path.cwd() / "experiments"
sys.path.insert(0, str(HERE))
import protocol as proto    # the robot runs of the paper (configs/protocol.yaml), checked in simulation
import run_box as rb        # start/goal -> plan -> simulation -> pre-flight; --execute moves the robot (not used here)
import run_plan as rp       # config, planner, executor in simulation, Swift replay

from pandaSim.control.contact_qp import FER_QD_MAX
from pandaSim.control.primitive_executor import contact_pose
from pandaSim.control.viewer import _scene
from pandaSim.planning import se3

NOTEBOOK = True          # Swift inside the output cell (False: a browser tab)
REPLAY_SPEED = 5.0       # a replay plays this many times faster than the run
RUNS = HERE / "runs" / "notebook"          # replay logs of this notebook (*.npz, not in git)
PROTOCOL = yaml.safe_load((HERE / "configs" / "protocol.yaml").read_text())
OKABE_ITO = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9", "#000000"]
STYLES = ["-", "--", "-", "--", "-", "--", "-"]


def task(exp, name):
    # (start, goal) of a task of the protocol
    t = PROTOCOL["experiments"][exp]["tasks"][name]
    return t["start"], t["goal"]


def config(grasp="free", plan_grasp="free", overrides=None, lead=False):
    # box_real.yaml with the grasp models (free: bearing fingertips, rigid: original fingers) and overrides such as
    # {"planner.gait.beta_deg": 25, "box.mass": 0.8}; a key that is not in the config raises KeyError
    cfg = rp.load_config(rb.REAL_CONFIG, grasp == "free", plan_grasp == "free")
    for key, value in (overrides or {}).items():
        *path, last = key.split(".")
        d = cfg
        for k in path:
            d = d[k]
        if last not in d:
            raise KeyError(f"{key} is not in the config")
        d[last] = value
    cfg["execution"]["robot"]["lead"] = lead
    return cfg


def simulate(start, goal, gait=False, grasp="free", plan_grasp="free", speed=0.3, lead=False, tracking=False,
             expect_stop=False, overrides=None, quiet=False, chain=False):
    # plan, simulate and pre-flight one task exactly as run_box.py does (no robot); poses are "x,y,yaw_deg,face".
    # gait=True: run_box.py --gait; chain=True: run_box.py --chain (two pivots in one grasp)
    cfg = config(grasp, plan_grasp, overrides, lead)
    res = rb.plan_and_simulate(cfg, rb.parse_pose(start), rb.parse_pose(goal), speed, tracking=tracking, gait=gait,
                               chain=chain)
    res.update(cfg=cfg, preflight=rb.preflight(res, cfg, expect_stop), speed=speed)
    if not quiet:
        print(f"{start} -> {goal}   {'gait' if gait else 'chain' if chain else 'sequencer'}, planned with {plan_grasp}, "
              f"executed with {grasp}, speed {speed}")
        rp.report(res)
    return res


def plan_only(start, goal, gait=False, grasp="free", overrides=None):
    # the planner alone (its rollouts of the candidate steps included), no execution of the plan
    cfg = rp.scale_speed(config(grasp, grasp, overrides), 1.0)
    robot, box = rp.make_robot(cfg), rp.make_box(cfg)
    T0, Tg = (rp.box_pose(box, rb.parse_pose(p)) for p in (start, goal))
    return rp.make_planner(cfg, robot, gait).plan(box, T0, Tg)


def replay(res, name, speed=REPLAY_SPEED, window=None):
    # replay a simulated run in Swift; window=(t0, t1) plays only that part (s of the run)
    path = rp.save({**res, "task": name}, res["cfg"], RUNS)
    return rp.show(path, speed, window=window, notebook=NOTEBOOK)


def step_window(res, k):
    # (t0, t1) of step k of a run: from its first segment (pre) to its last (ready)
    segs = [s for s in res["log"].segments if s.get("step") == k]
    return segs[0]["t0"], segs[-1]["t_end"]


def show_poses(start, goal, cfg=None):
    # static Swift view: the robot at q_ready, the box at the start, its goal and the marked corner
    cfg = cfg or config()
    box = rp.make_box(cfg)
    T0, Tg = (rp.box_pose(box, rb.parse_pose(p)) for p in (start, goal))
    base = se3.rp(np.eye(3), np.asarray(cfg["robot"]["base"], float))
    env, _, _ = _scene(base, rp.scene_objects(T0, Tg, box.size), np.r_[cfg["robot"]["q_ready"], 0.0], notebook=NOTEBOOK)
    env.step(0.05)
    return env


def describe(plan):
    # every step of a plan: kind, grasp width, the screw motions in one grasp, phi limits, gait vertex
    if not plan.found:
        print("no feasible plan")
        return
    print(f"cost {plan.cost:.3f}, {len(plan.steps)} step(s), planning {plan.stats.get('time_s', 0):.1f} s")
    for i, st in enumerate(plan.steps):
        pl = st.plan
        print(f"step {i}: {st.kind}, grasp width {pl.width * 1e3:.0f} mm, {len(pl.screws)} screw motion(s) in one grasp"
              + (f", vertex at {np.round(pl.vertex, 3)}" if hasattr(pl, "vertex") else ""))
        for S, th in pl.screws:
            S = np.asarray(S, float)
            if np.linalg.norm(S[:3]) > 0.5:          # rotation about an axis (pitch 0 here)
                p = np.cross(S[:3], S[3:])            # point of the axis closest to the origin
                print(f"    rotation {math.degrees(th):+6.1f} deg about {np.round(S[:3], 2)} through {np.round(p, 3)}")
            else:
                print(f"    translation {th * 1e3:6.1f} mm along {np.round(S[3:], 2)}")
        if getattr(pl, "phi_lim", None):
            print(f"    phi limits {np.round(np.degrees(pl.phi_lim), 1)} deg (palm clear of the box)")


def footprint(T, size):
    # the four corners of the face on the table, in order around it (xy)
    c = np.array([[x, y, z] for x in (-.5, .5) for y in (-.5, .5) for z in (-.5, .5)]) * np.asarray(size)
    P = c @ T[:3, :3].T + T[:3, 3]
    low = P[P[:, 2] < P[:, 2].min() + 1e-6][:, :2]
    m = low.mean(axis=0)
    return low[np.argsort(np.arctan2(*(low - m).T[::-1]))]


def plot_top(res, ax=None):
    # top view of the plan: the box after each step, the goal, the marked corner, reach and keep-out of the base
    ax = ax or plt.subplots(figsize=(5.5, 5.5))[1]
    size = res["box"].size
    poses = [res["T0"]] + ([s.T_after for s in res["plan"].steps] if res["plan"].found else [])
    for i, T in enumerate(poses):
        f = footprint(T, size)
        ax.fill(*f.T, color=OKABE_ITO[0], alpha=0.12 + 0.5 * i / max(len(poses) - 1, 1), ec=OKABE_ITO[0], lw=0.8)
        ax.text(*T[:2, 3], str(i), ha="center", va="center", fontsize=8)
        ax.plot(*(T @ np.r_[np.asarray(rb.MARK) * np.asarray(size) / 2, 1])[:2], "o", color=OKABE_ITO[3], ms=4)
    f = footprint(res["Tg"], size)
    ax.fill(*f.T, fill=False, ec=OKABE_ITO[2], lw=1.5, ls="--", label="goal")
    for st in (res["plan"].steps if res["plan"].found else []):
        if hasattr(st.plan, "vertex"):
            ax.plot(*st.plan.vertex[:2], "x", color="k", ms=7)
    for r, ang, lab in ((0.30, -40, "keep-out of the base"), (0.78, -22, "reach")):
        ax.add_patch(plt.Circle((0, 0), r, fill=False, ls=":", color="0.5"))
        ax.text(r * math.cos(math.radians(ang)), r * math.sin(math.radians(ang)), lab, fontsize=7, color="0.4",
                rotation=90 + ang, ha="center", va="bottom")
    ax.plot([], [], "o", color=OKABE_ITO[3], ms=4, label="marked corner")
    ax.set(xlabel="x (m)", ylabel="y (m)", aspect="equal", xlim=(0.2, 0.8), ylim=(-0.35, 0.35),
           title="box after each step (0: start)")
    ax.legend(loc="upper left", fontsize=7, frameon=False)
    return ax


def plot_run(res, title=""):
    # joint speeds against the bound of the QP, joint angles in their range, and phi with its limits in each pivot
    log = res["log"]
    a, lb, ub = log.arrays(), *res["robot_limits"]
    bound = res["cfg"]["execution"]["qp"]["qd_scale"]
    fig, ax = plt.subplots(3, 1, figsize=(9, 6.5), sharex=True)
    for j in range(7):
        ax[0].plot(a["t"], np.abs(a["qd"][:, j]) / FER_QD_MAX[j], color=OKABE_ITO[j], ls=STYLES[j], lw=0.9, label=f"{j + 1}")
        ax[1].plot(a["t"], (a["theta"][:, j] - lb[j]) / (ub[j] - lb[j]), color=OKABE_ITO[j], ls=STYLES[j], lw=0.9)
    ax[0].axhline(bound, color="k", ls=":", lw=0.8)
    ax[0].text(a["t"][0], bound, " bound of the QP", va="bottom", fontsize=7)
    ax[0].set_ylabel("|joint speed|\n/ limit")
    ax[0].legend(title="joint", ncol=7, fontsize=7, title_fontsize=7, frameon=False, loc="upper right")
    ax[1].set(ylabel="joint angle\n(0, 1: limits)", ylim=(-0.03, 1.03))
    ax[2].plot(a["t"], np.degrees(a["phi"]), color=OKABE_ITO[0], lw=1.0)
    ax[2].set(ylabel="phi (deg)", xlabel="time (s)")
    for s in log.segments:
        if s["name"] in ("pivot", "slide"):
            for x in ax:
                x.axvspan(s["t0"], s["t_end"], color="0.85" if s["name"] == "slide" else "#cfe3f3", lw=0)
        lim = getattr(res["plan"].steps[s["step"]].plan, "phi_lim", None) if s["name"] == "pivot" else None
        if lim:
            ax[2].hlines(np.degrees(lim), s["t0"], s["t_end"], color=OKABE_ITO[3], ls="--", lw=0.9)
        if not s["ok"]:
            for x in ax:
                x.axvline(s["t_end"], color=OKABE_ITO[3], lw=1.2)
            ax[0].text(s["t_end"], 0.9, f" {s['cause']}", color=OKABE_ITO[3], fontsize=8, transform=ax[0].get_xaxis_transform())
    fig.suptitle(title + "   (blue band: pivot or gait screw, grey: slide; dashed: phi limits)", fontsize=9)
    fig.tight_layout()
    return fig


print("pandaSim tour ready; config", rb.REAL_CONFIG.name, "| Swift", "in the cell" if NOTEBOOK else "in a browser tab")

## 1. The box, its faces and the marked corner

The box is 0.15 × 0.06 × 0.25 m (`box.size` along its x, y, z axes). The fingers span only the 0.06 m side, so the box
is never grasped, or put, on its largest face. A pose is written `x,y,yaw_deg,face`: the centre on the table, the yaw
about the vertical and the face on the table:

| face | on the table | pitch about the box y axis |
|---|---|---|
| `lying` | the 0.06 × 0.25 face | −90° |
| `lying_flip` | the opposite face | +90° |
| `standing` | the 0.15 × 0.06 face | 0° |
| `standing_flip` | the opposite face | 180° |

The real box carries a mark on one corner, the (+x, +y, +z) corner of the box frame (`MARK`). The box is therefore
**not symmetric** (`planner.symmetric: false`): a pose turned by 180° is another pose, the yaw runs over 360°, and the
planner must reach the goal itself, not one of the four poses an unmarked box cannot be told apart in.

The view below shows the four faces side by side at yaw 0, each with its mark.

In [ ]:
cfg = config()
box = rp.make_box(cfg)
objects = []
for (face, pitch), y in zip(rb.FACES.items(), (-0.27, -0.09, 0.09, 0.27)):
    T = rp.box_pose(box, rb.parse_pose(f"0.55,{y},0,{face}"))
    objects += [dict(type="cuboid", scale=list(box.size), pose=T, color=[0.1, 0.3, 0.9, 0.8]),
                dict(type="sphere", radius=0.01, pose=T @ se3.rp(np.eye(3), np.asarray(rb.MARK) * np.asarray(box.size) / 2), color="red")]
    print(f"{face:14s} y = {y:+.2f}: centre height {T[2, 3]:.3f} m, pose text back: {rb.pose_text(T)}")
base = se3.rp(np.eye(3), np.asarray(cfg["robot"]["base"], float))
env_faces, _, _ = _scene(base, objects, np.r_[cfg["robot"]["q_ready"], 0.0], notebook=NOTEBOOK)
env_faces.step(0.05)

A task is a start and a goal pose. Change them and run the cell to see where they are before planning anything
(the table area for the robot runs is x 0.30 to 0.75 m, |y| < 0.35 m; box corners stay 0.30 m from the base axis).

In [ ]:
START, GOAL = "0.50,0.00,0,lying", "0.55,0.10,30,standing"
env_poses = show_poses(START, GOAL)

## 2. Primitive motions: slide and pivot (paper Sec. 7.2)

Each primitive is one grasp and one or more **constant screw motions** of the box, all known before the motion
(`planning/box_primitives.py`):

- **Slide**: the box stays on its face and moves on the table (translation and yaw). Planar screws, tracked with pose
  feedback. Checked for the tipping margin (friction, mass).
- **Pivot**: a rotation by 90° about the bottom edge of the box that lies on the table (pitch-0 screw), grasped 0.02 m
  inside the top and far faces. Feed-forward along the screw: the twist of the contact frame is constant in the frame
  and scaled by the quintic time scaling $\dot s(t)$.

The executor runs every step the same way (`run_plan`): `pre` (0.10 m above the grasp) → `approach` → grasp (20 N) →
the screw motions → release → `retreat` → `ready` (joint move back to `q_ready`). Below: the two tasks of Sec. 7.2 at
the speed of the robot runs (0.3). The simulation includes the joint and speed limits of the arm, the QP and the
clearance of arm, hand and box.

In [ ]:
slide = simulate(*task("prim", "slide"))
describe(slide["plan"])

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 5))
plot_top(slide, ax)
plot_run(slide, "slide")
plt.show()
env_slide = replay(slide, "prim_slide")

In [ ]:
pivot = simulate(*task("prim", "pivot"))
describe(pivot["plan"])
plot_run(pivot, "pivot")
plt.show()
env_pivot = replay(pivot, "prim_pivot")

## 3. Inside a pivot: contact QP and the bearing angle $\phi$

The fingertips carry a bearing, so during a pivot the hand may turn about the finger axis $\hat n$ relative to the box
(bearing angle $\phi$). `ContactQP` (paper Sec. 6, eq. qp_a–d) solves for the 7 joint speeds and $\dot\phi$:

- **equality** $J^{aug}_c(\theta, \phi)\,[\dot\theta; \dot\phi] = \mathcal V_c$: the contact frame follows the planned
  twist exactly (hard: the QP is infeasible rather than off the path);
- **cost**: joints towards the middle of their range, manipulability up, small $\dot\phi$;
- **inequalities**: damper near the joint limits ($\rho_s$, $\rho_e$ = 0.13, 0.40 rad on the robot), joint speeds
  at most `qd_scale` × the libfranka limits (0.8 × the speed factor of the run), and $\phi$ inside the interval in
  which the palm clears the box (`clearance.phi_limits`; hand model `box`: its bounding box, the cautious model used on
  the robot; `mesh`: its collision mesh).

The rigid grasp is the same QP without the column of $\phi$ (7 unknowns; Sec. 6 runs it). The cell below solves
the QP again at five instants of the pivot of Sec. 2, from the logged joints and $\hat\phi$, and compares with the
joint speeds the executor sent: the twist is the constant body twist of the pivot times $\dot s(t)$ of the quintic
time scaling. The bearing is used most in the middle of the pivot.

In [ ]:
from pandaSim.control.contact_qp import ContactQP
from pandaSim.planning import clearance
from pandaSim.planning.primitives import screw_twist, time_scaling

robot, st = pivot["robot"], pivot["plan"].steps[0]
pl, (S, theta) = st.plan, st.plan.screws[0]
T_c_obj = se3.inv(pl.T_grasp) @ pl.T_start
for model in ("box", "mesh"):
    print(f"phi limits, hand model {model:4s}: {np.round(np.degrees(clearance.phi_limits(T_c_obj, pivot['box'].size, model=model)), 1)} deg")

a, segs = pivot["log"].arrays(), pivot["log"].segments
j = next(j for j, s in enumerate(segs) if s["name"] == "pivot")
seg, idx = segs[j], np.flatnonzero(a["segment"] == j)
q_at = lambda k: np.r_[a["theta"][k], a["phi"][k]]
V1 = screw_twist(S, theta, contact_pose(robot, q_at(idx[0])))     # body twist of {c} for s_dot = 1, constant
free = ContactQP(robot, free_rotation=True, phi_lim=pl.phi_lim, **pivot["cfg"]["execution"]["qp"])
print(f"pivot by {math.degrees(theta):.0f} deg in {seg['Tf']:.1f} s; J_aug {free.jacobian(q_at(idx[0])).shape}, "
      f"speed bound {np.round(free.qd_max, 2)} rad/s")
print(f"{'s':>5s} {'phi (deg)':>9s} {'phi_dot':>8s} {'max |qd|/limit':>14s} {'QP - logged':>11s}")
for frac in (0.1, 0.3, 0.5, 0.7, 0.9):
    k = idx[int(frac * (len(idx) - 1))]
    s, sdot = time_scaling(a["t"][k] - seg["t0"], seg["Tf"])
    qd = free.solve(q_at(k), sdot * V1)
    print(f"{s:5.2f} {math.degrees(a['phi'][k]):9.1f} {qd[7]:+8.3f} {np.abs(qd[:7] / FER_QD_MAX).max():14.3f} "
          f"{np.abs(qd[:7] - a['qd'][k, :7]).max():11.1e}")

## 4. Sequencing with a marked box (paper Sec. 7.4)

`Sequencer` searches the template `[slide] → pivots A → [slide] → pivots B → goal` (at most 3 pivots). Cheap tests
first (keep-out around the base, reach of every grasp point, tipping margin of the slides), then the candidates are
ranked by the quasi-static work (slides: friction × path, pivots: weight × rise of the centre, 0.3 per regrasp), and
the cheapest are **rolled out with the executor** (`Rollout`, the same code as above): the first one that passes is the
plan. So a plan is only returned if the executor can run it.

The three tasks of Sec. 7.4: `turn` (onto the adjacent face), `flip` (onto the opposite face) and `flip_turn`. Pick one
(`flip` takes about a minute to plan and simulate).

In [ ]:
SEQ_TASK = "turn"          # "turn", "flip" or "flip_turn"
seq = simulate(*task("seq", SEQ_TASK))
describe(seq["plan"])
fig, ax = plt.subplots(1, 1, figsize=(5, 5))
plot_top(seq, ax)
plot_run(seq, f"sequencing: {SEQ_TASK}")
plt.show()
env_seq = replay(seq, f"seq_{SEQ_TASK}")

**Why the mark matters.** With `planner.symmetric: true` (an unmarked box, the simulation config `box_plan.yaml`), any
of the four poses the box looks the same in counts as the goal; with the mark only the goal itself does. The planner
alone (no execution) for a few goals, both ways:

In [ ]:
for start, goal in [task("seq", "flip"), ("0.50,0.00,0,lying", "0.55,0.05,180,lying")]:
    for sym in (True, False):
        p = plan_only(start, goal, overrides={"planner.symmetric": sym})
        kinds = " -> ".join(s.kind for s in p.steps) if p.found else "no plan"
        print(f"{start} -> {goal}   symmetric={sym!s:5s}: {kinds:32s} cost {p.cost:.2f}")

## 5. Object gaiting with one arm (paper Sec. 7.3)

The box stays on its face and walks on its vertices, without sliding. One **gait step** is one grasp at the far end of
the support face and five screw motions whose axes all pass through a vertex $\boldsymbol p_j$:

1. tilt by $\gamma$ (15°) about the short edge that contains $\boldsymbol p_j$ (the box stands on that edge),
2. tilt by $\beta$ (30°) about the box edge through $\boldsymbol p_j$ (the box now stands on $\boldsymbol p_j$ alone),
3. turn by $\alpha_j$ about the vertical through $\boldsymbol p_j$,
4. back by $\beta$, 5. back by $\gamma$.

Net: a planar rotation by $\alpha_j$ about $\boldsymbol p_j$. Release, regrasp at the other end, and the next step turns
about a vertex there. The planner (`GaitPlanner`) chooses the vertices and angles that reach the goal **exactly** with
the fewest steps, then the smallest $\lVert\alpha\rVert$, with $|\alpha_j| \le \alpha_{\max}$ (45°), and rolls every
step out like the sequencer does. First the angles alone (`solve_angles`):

In [ ]:
from pandaSim.planning import gait

GAIT_TASK = "lying"        # "lying" (3 steps) or "standing" (4 steps)
g_cfg = config()
box = rp.make_box(g_cfg)
T0, Tg = (rp.box_pose(box, rb.parse_pose(p)) for p in task("gait", GAIT_TASK))
gp = g_cfg["planner"]["gait"]
sols = gait.solve_angles(T0, Tg, box.size, k_max=gp["k_max"], alpha_max=math.radians(gp["alpha_max_deg"]))
print(f"{len(sols)} solutions; the best few (vertex = (end, side) of the support face):")
for seq_v, alphas in sols[:5]:
    print(f"  k = {len(alphas)}: vertices {list(seq_v)}, alpha {np.round(np.degrees(alphas), 1)} deg")

In [ ]:
walk = simulate(*task("gait", GAIT_TASK), gait=True)
describe(walk["plan"])
fig, ax = plt.subplots(1, 1, figsize=(5, 5))
plot_top(walk, ax)          # x: the vertex of each step
plot_run(walk, f"gait: {GAIT_TASK}")
plt.show()

The whole walk, then step 0 alone at the real speed, to look at the five screw motions about the vertex (orbit the
view to see the box stand on one vertex during the turn).

In [ ]:
env_walk = replay(walk, f"gait_{GAIT_TASK}")

In [ ]:
env_step = replay(walk, f"gait_{GAIT_TASK}_step0", speed=1.0, window=step_window(walk, 0))

## 6. Free rotation against the rigid grasp (paper Sec. 7.5)

The same pivot, **planned with free rotation**, executed with the bearing fingertips (free) and with the original
fingers (rigid: the hand is fixed to the box). Without the bearing the arm must turn the hand with the box; near the
singularity the hard equality of the QP becomes infeasible and the run stops part-way (`predicted_stop`). These runs
are executed on the robot too, up to the stop (`--expect-stop`): the paper shows the rigid failures in hardware.

Safety finding behind the speed: where the rigid QP gives up, it first drives joints to its speed bound. The speed
factor therefore also scales that bound, the experiment runs at 0.15, and the pre-flight refuses an expected stop after
which the hand would coast more than 5 mm (the lag of the joint controller). Since the review (D4) the speed factor
also scales the QP's rates (centering K, manipulability lam, damper eta), so the planner's rollouts at speed 1 and the
run at 0.15 follow the same joint path; the rigid stops are 52 deg (c, d) and 55 deg (e) of 90.

In [ ]:
FR_TASK = "c"              # "a" to "e": rigid passes in a and b, stops in c, d, e
fr_speed = PROTOCOL["experiments"]["fr"]["speed"]
fr = {g: simulate(*task("fr", FR_TASK), grasp=g, speed=fr_speed) for g in ("free", "rigid")}
stop = rb.predicted_stop(fr["rigid"]["log"], fr["rigid"]["plan"].steps)
print("\nrigid:", f"stops in step {stop['step']} at {stop['angle_deg']:.0f} of {stop['of_deg']:.0f} deg" if stop else "no stop")

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(9, 4.5), sharex=True)
for g, ls in (("free", "-"), ("rigid", "--")):
    a = fr[g]["log"].arrays()
    piv = [s for s in fr[g]["log"].segments if s["name"] == "pivot"][0]
    m = (a["t"] >= piv["t0"] - 1) & (a["t"] <= piv["t0"] + piv["Tf"] + 1)
    t = a["t"][m] - piv["t0"]
    ax[0].plot(t, (np.abs(a["qd"][m, :7]) / FER_QD_MAX).max(axis=1), ls=ls, color="k", label=g)
    lb, ub = fr[g]["robot_limits"]
    ax[1].plot(t, np.minimum(a["theta"][m] - lb, ub - a["theta"][m]).min(axis=1), ls=ls, color="k", label=g)
ax[0].axhline(fr["rigid"]["cfg"]["execution"]["qp"]["qd_scale"], color=OKABE_ITO[3], ls=":", lw=1)
ax[0].set_ylabel("largest |joint speed|\n/ limit")
ax[1].set(ylabel="smallest distance to\na joint limit (rad)", xlabel="time from the start of the pivot (s)")
ax[0].legend(frameon=False)
fig.suptitle(f"pivot {FR_TASK}: free rotation (solid) against the rigid grasp (dashed); dotted: speed bound of the QP", fontsize=9)
fig.tight_layout()
plot_run(fr["rigid"], f"rigid grasp, task {FR_TASK}")
plt.show()

In [ ]:
print("pre-flight of the rigid run as run_box.py --rigid" + (" --expect-stop" if stop else "") + ":")
rb.print_preflight(rb.preflight(fr["rigid"], fr["rigid"]["cfg"], expect_stop=stop is not None))
env_rigid = replay(fr["rigid"], f"fr_{FR_TASK}_rigid", speed=2.0)

## 6b. Two pivots in one grasp (`run_box.py --chain`)

With the bearing, the box can be pivoted twice in succession without a regrasp: a roll by $\pi$ about the finger axis
onto the opposite face, 0.40 m further on. The bearing takes only part of the $\pi$ (the arc of $\phi$ in which the palm
clears the box, ~108 deg); the hand turns by the rest. That needs a grasp near an edge: the study
`experiments/double_pivot_study.py` found one region with margins, the back-bottom corner, 10 mm from the back face and
27.5 mm above the table, with the hand tilted 68 deg behind the box (`planning/chain.py`). The planner's usual corner
grasp has no end pose clear of the table. The pre pose and the retreat are 0.25 m above the grasp (0.10 m would sweep
the fingers through the box).

The experiment `chain` of the protocol: the same roll with the bearing (free) and with the original fingers (rigid),
at speed 0.15. Free rotation completes it; the rigid grasp stops in the second pivot (entry a: 2 deg into it, b: 81 deg).

In [ ]:
CHAIN_TASK = "a"           # "a" or "b"
ch_speed = PROTOCOL["experiments"]["chain"]["speed"]
chain = {g: simulate(*task("chain", CHAIN_TASK), chain=True, grasp=g, speed=ch_speed, quiet=True) for g in ("free", "rigid")}
describe(chain["free"]["plan"])
for g, res in chain.items():
    done = sum(1 for s in res["log"].segments if s["name"] == "pivot" and s["ok"])      # pivots completed
    stop = rb.predicted_stop(res["log"], res["plan"].steps)
    reached = 90.0 * done + (stop["angle_deg"] if stop else 0.0)
    print(f"{g:5s}: {'completed' if res['log'].ok else 'stopped (' + next(s['cause'] for s in res['log'].segments if not s['ok']) + ')'}, "
          f"box rolled by {reached:.0f} of 180 deg")
plot_run(chain["free"], f"chain {CHAIN_TASK}, free rotation: both pivots in one grasp")
plt.show()

In [ ]:
print("pre-flight of the rigid run as run_box.py --chain --rigid --expect-stop:")
rb.print_preflight(rb.preflight(chain["rigid"], chain["rigid"]["cfg"], expect_stop=True))
env_chain = replay(chain["free"], f"chain_{CHAIN_TASK}_free")

In [ ]:
env_chain_rigid = replay(chain["rigid"], f"chain_{CHAIN_TASK}_rigid", speed=3.0)

## 7. Pre-flight and the lag of the joint controller

Before any robot run, `run_box.py` prints the pre-flight list (hard checks block the run; warnings do not): table area,
keep-out, plan found, every check of the simulation, observability of $\phi$ (the box cannot turn about the finger axis
on its own), joint-limit margin and panda-py's joint walls, arm clearance, payload, force threshold, accelerations.

The ideal simulation commands the joints exactly. On the robot, panda-py's joint controller makes the measured joints
lag the target by $K_d/K_p \cdot \dot q$ (`PANDA_LAG`). `simulate_tracking` runs the plan once more with that lag;
`--lead` compensates it. The gait has 25 screw motions in a row, so it collects the most lag:

In [ ]:
from pandaSim.control.backends import PANDA_LAG

print("lag per joint [s]:", np.round(PANDA_LAG, 3))
for lead in (False, True):
    tr = rb.simulate_tracking(walk, walk["cfg"], lead=lead)
    print(f"gait with the lag, lead={lead!s:5s}: completed {tr['ok']}, object error {tr['err_mm']:.1f} mm, "
          f"{tr['err_deg']:.2f} deg, hand off the screw arcs by up to {tr['pivot_dev_mm']:.1f} mm")

In [ ]:
print("pre-flight of the gait run (without the lag simulation; run_box.py adds it):")
rb.print_preflight(walk["preflight"])

Since the review (D9), a gait that misses its goal in the lag simulation is refused without `--lead`, so the gait
experiment of `protocol.yaml` sets `lead: true` (the global `lead` stays false). These commands are valid only after
the air run on the robot has shown that the measured lag matches the model (`docs/HARDWARE_DAY.md`).

## 8. The robot runs of the paper (protocol)

`configs/protocol.yaml` lists every robot run of Sec. 7, in the order of the paper; `protocol.py` plans and simulates
each entry exactly as `run_box.py` does and prints the commands. The last verdicts (`results/protocol.json`):

In [ ]:
import json

verdicts = {r["name"]: r for r in json.loads((HERE / "results" / "protocol.json").read_text())}
print(f"{'entry':20s} {'reps':>4s} {'speed':>5s}  {'verdict':7s} {'plan':38s} duration")
for exp, name, grasp, spec in proto.entries(PROTOCOL):
    r = verdicts.get(f"{exp}_{name}_{grasp}")
    if r is None:
        print(f"{exp}_{name}_{grasp:14s} not checked yet: run protocol.py")
        continue
    stop = f"  stop at {r['stop']['angle_deg']:.0f} of {r['stop']['of_deg']:.0f} deg" if r["stop"] else ""
    print(f"{exp}_{name}_{grasp:{18 - len(exp) - len(name)}s} {spec['repetitions']:4d} {spec['speed']:5.2f}  "
          f"{'GO' if r['go'] else 'NO-GO':7s} {' -> '.join(r['steps']):38s} {r['duration_s']:4.0f} s{stop}")

One entry checked again here, and the command of its first repetition (printed only: you run it at the robot, after
the ladder of `docs/HARDWARE_DAY.md`; it moves the robot only after its own checks and a typed confirmation).

In [ ]:
EXP, NAME, GRASP = "fr", "d", "rigid"
spec = next(s for e, n, g, s in proto.entries(PROTOCOL) if (e, n, g) == (EXP, NAME, GRASP))
r = proto.check_entry(PROTOCOL, spec, GRASP)
print("GO" if r["go"] else "NO-GO", r["steps"], f"{r['duration_s']:.0f} s", r["stop"] or "", r["failed"] or "")
print(proto.command(PROTOCOL, spec, r["flags"], f"{EXP}_{NAME}_{GRASP}_r1", "172.16.0.2"))

## 9. After a robot run

Each robot run writes `experiments/runs/box/<name>/`: `plan.json`, `expected.npz` (the simulation), `preflight.json`,
`real.npz` (the robot: joints, $\hat\phi$, wrench, timing) and `compare.json`. Then, outside this notebook:

| Step | Command |
|---|---|
| final pose of the box, by touching three corners (read only) | `python experiments/calibrate_box.py 172.16.0.2 --measure experiments/runs/box/<name>` |
| plan against measurement, one run | `python experiments/analyze_run.py experiments/runs/box/<name>` |
| panels (b), (c) of the pivot figure | `python experiments/paper_figure.py experiments/runs/box/<name> --out <Figures>/exp_pivot` |
| rows of the tables of Sec. 7 | `python experiments/hardware_tables.py` |
| parameters of the paper against the code | `python experiments/paper_numbers.py` |

The robot runs of the protocol found so far:

In [ ]:
import hardware_tables as ht

data = ht.collect(PROTOCOL)
done = {k: v for k, v in data.items() if v}
print(f"{sum(len(v) for v in data.values())} robot runs of the protocol found in {ht.RUNS}")
for (exp, name, grasp), recs in done.items():
    print(f"  {exp}_{name}_{grasp}: " + ", ".join(f"{r['outcome']} ({r['error_mm']:.1f} mm)" if r["error_mm"] is not None
                                                 else r["outcome"] for r in recs))

## 10. Playground: your own task and parameters

Change anything in the next cell and run it, then the two after it. The parameters:

- `START`, `GOAL`: `x,y,yaw_deg,face`, face in `lying`, `lying_flip`, `standing`, `standing_flip`.
- `GAIT`: `True` for object gaiting (the goal must be on the same face), `False` for the sequencer.
- `CHAIN`: `True` for two pivots in one grasp (the goal is the start rolled by pi onto the opposite face, 0.40 m
  further along the roll direction; see Sec. 6b).
- `GRASP`: execution model, `"free"` (bearing) or `"rigid"` (original fingers); `PLAN_GRASP`: the planner's model.
- `SPEED`: speed factor of the execution (0.3 on the robot, at most 0.5 at level `low`); it also scales the speed
  bound of the QP.
- `LEAD`: lag compensation (only affects the lag simulation, `TRACKING = True`).
- `OVERRIDES`: any value of `configs/box_real.yaml` by its path, for example
  `"planner.gait.gamma_deg"`, `"planner.gait.beta_deg"`, `"planner.gait.alpha_max_deg"`, `"planner.gait.k_max"`,
  `"planner.max_pivots"`, `"planner.symmetric"`, `"planner.hand_model"` (`"box"` or `"mesh"`), `"box.mass"`, `"box.mu"`,
  `"box.size"`, `"execution.grasp.force"`, `"execution.qp.rho_s"`, `"execution.qp.lam"`, `"execution.pivot.w_max"`.

In [ ]:
START = "0.45,0.15,0,lying"
GOAL = "0.60,-0.10,120,standing"
GAIT = False
CHAIN = False
GRASP, PLAN_GRASP = "free", "free"
SPEED = 0.3
LEAD, TRACKING = False, False
OVERRIDES = {
    "planner.hand_model": "box",
    "planner.symmetric": False,
    "planner.gait.alpha_max_deg": 45.0,
    "box.mass": 1.0,
}

env_mine = show_poses(START, GOAL, config(GRASP, PLAN_GRASP, OVERRIDES))

In [ ]:
mine = simulate(START, GOAL, gait=GAIT, chain=CHAIN, grasp=GRASP, plan_grasp=PLAN_GRASP, speed=SPEED, lead=LEAD, tracking=TRACKING,
                expect_stop=False, overrides=OVERRIDES)
if mine["plan"].found and GRASP == "rigid" and rb.predicted_stop(mine["log"], mine["plan"].steps):
    mine["preflight"] = rb.preflight(mine, mine["cfg"], expect_stop=True)     # as run_box.py --expect-stop
print()
rb.print_preflight(mine["preflight"])
describe(mine["plan"])
if mine["plan"].found:
    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    plot_top(mine, ax)
    plot_run(mine, "my task")
    plt.show()

In [ ]:
STEP = None                # None: the whole run; k: only step k, at the real speed
if mine["log"] is not None:
    env_mine_run = (replay(mine, "mine") if STEP is None
                    else replay(mine, f"mine_step{STEP}", speed=1.0, window=step_window(mine, STEP)))

The same task from the command line (pre-flight, run folder in `experiments/runs/box/`; `--replay` opens Swift in a
browser tab), as you would run it before the robot:

```bash
.venv/bin/python experiments/run_box.py --start 0.45,0.15,0,lying --goal 0.60,-0.10,120,standing --level low --speed 0.3
```

Add `--gait`, `--chain`, `--rigid`, `--expect-stop`, `--lead` as above. `--execute --host 172.16.0.2` is the only flag that moves
the robot.

## 11. What the code review of 2026-11-24/25 changed

A critic and a defender argued every design decision, and a code critic and a reviewer every piece of code, in one
thread per item (`~/pandaSim/pandaSim_handoff/review_2026-11-24/DEBATE.md`; summary: `FINAL_REPORT.md` there). Every
change passed three gates: pytest, a capture of the reference behaviour (`refcheck.py`: plans, simulated runs and
pre-flight verdicts of the 17 protocol runs and the 10 sim tasks), and, for the robot path, only bug fixes or provably
identical changes tested with fakes. No robot was used.

- **Legacy code removed** (about 16,700 lines: old controllers, adapters, Genesis examples, old notebooks; the
  `genesis` extra). Behaviour identical; the old code is on `master` and under the tag `pre-cleanup-2026-11-24`.
- **Bugs fixed**: the hand model leaked between planners in one process; the config's speeds were overwritten by
  `scale_speed`; after a gripper call the robot backend returned the joint state from before the call; the predicted stop
  assumed quintic time scaling; a segment record could be lost if the zero command raised.
- **Sounder**: the planner checks arm collisions at every sample of a rollout (links moved up to 13 mm between checks).
- **Faster**: planning about -21 %, the simulated runs about -30 %, the 1 kHz tick 0.12 -> 0.10 ms (sphere pre-tests for
  collisions, the QP built in place, analytic gradients for the gait angles, `se3` instead of pytransform3d; outputs
  identical).
- **Design fixes**: the speed factor scales the QP's rates, so the planner's rollouts and the robot run follow the same
  joint path (D4); a grasp costs kappa in the planner's ranking, as eq:seq_a (D2; the cost only ranks plans, it does not
  act during the motion); a gait that misses its goal in the lag simulation needs `--lead` (D9); the final object error
  from forward kinematics is labelled "tracking (model)", not accuracy (D13).
- **New**: two pivots in one grasp (Sec. 6b).
- **On a separate branch, not merged**: `exp/heavy-box`, the 5 kg box (load model of the held box, load checks in the
  pre-flight, a wrench stop relative to the predicted load, one reflex setting per run). At 5 kg it makes 11 of 17 runs
  NO-GO until the box-table and pad friction and the deflection under load are measured at the robot.

The cell below shows the reference check: it captures the current behaviour and compares it with the reference taken
after the design fixes (about 2 min on 12 cores; set `RUN_REFCHECK = True`).

In [ ]:
import subprocess

RUN_REFCHECK = False
REVIEW = Path.home() / "pandaSim" / "pandaSim_handoff" / "review_2026-11-24"
if RUN_REFCHECK and (REVIEW / "refcheck.py").exists():
    repo, out = HERE.parent, RUNS / "refcheck_now.json"
    env = {k: v for k, v in __import__("os").environ.items() if k != "PYTHONPATH"} | {"PYTHONPATH": str(repo / "src")}
    subprocess.run([sys.executable, str(REVIEW / "refcheck.py"), "capture", str(repo), str(out), "--workers", "12"], env=env, check=True)
    print(subprocess.run([sys.executable, str(REVIEW / "refcheck.py"), "compare", str(REVIEW / "ref_after_design_024df85.json"), str(out)],
                         capture_output=True, text=True).stdout)